# Part A — Data Foundation
This notebook is the main runner only. All processing is implemented in `part_a.py`. Run cells in order from the repository root. Place supplied `generate_data.py` and `mock_llm.py` at the repository root and generate raw exports first.

## Task 1 — Reproducible setup
Create a virtual environment, install dependencies and freeze the actual versions. Run `python generate_data.py` from the repository root before the next cell. See README for numbered commands.

In [3]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent
if not (ROOT / "generate_data.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)
print("Root files:", sorted(p.name for p in ROOT.iterdir() if p.is_file()))

Project root: d:\Sowbarnika\Instamat_project
Root files: ['README.md', 'generate_data.py', 'requirements.txt', 'validate.py']


In [4]:
import subprocess, sys
subprocess.run([sys.executable, str(ROOT / "generate_data.py")], check=True)
raw_dir = ROOT/ "data" / "raw"
print("Raw files:", sorted(p.name for p in raw_dir.iterdir()))

Raw files: ['agent_scenarios.csv', 'customers.csv', 'orders.csv', 'policy_eval.csv', 'products.csv', 'tickets.csv']


In [5]:
from src.part_a import load_raw, audit, clean_customers, clean_orders, store, validate, run_sql, pandas_eda


D:\Sowbarnika\Instamat_project


## Task 2 — Raw audit
Inspect shapes, data types, missingness and the required dirty-data checkpoints without modifying raw files.

In [6]:
customers_raw, products_raw, orders_raw, tickets_raw = load_raw()
audit_table = audit(customers_raw, products_raw, orders_raw, tickets_raw)


 customers shape: (1200, 5)
Types:
 customer_id        str
city               str
signup_date        str
age            float64
gender             str
             missing   pct
customer_id        0  0.00
city               0  0.00
signup_date        0  0.00
age               61  5.08
gender            60  5.00

 products shape: (40, 4)
Types:
 product_id        str
product_name      str
category          str
price_inr       int64
              missing  pct
product_id          0  0.0
product_name        0  0.0
category            0  0.0
price_inr           0  0.0

 orders shape: (13084, 9)
Types:
 order_id                str
customer_id             str
product_id              str
quantity              int64
order_ts                str
status                  str
discount_pct          int64
order_amount            str
delivery_minutes    float64
                  missing    pct
order_id                0   0.00
customer_id             0   0.00
product_id              0   0.00
quantity  

## Task 3 — Clean customers
Normalize city and gender, impute invalid or missing ages using city medians, and parse signup dates.

In [7]:
customers = clean_customers(customers_raw)


Cities:
 city
Pune         227
Delhi        210
Bengaluru    197
Chennai      191
Hyderabad    191
Mumbai       184 
Genders:
 gender
female     610
male       560
unknown     30


## Task 4 — Clean orders
Parse timestamps, remove exact duplicates, retain latest order status, reconstruct invalid amounts and impute completed delivery times by category.

In [8]:
orders = clean_orders(orders_raw, products_raw)


Exact duplicates removed: 119
Older status records removed: 656
Amounts reconstructed: 112
Negative delivery values cleared: 105
Completed delivery values imputed: 774
Final status counts:
 status
completed    11110
cancelled     1199


## Task 5 — Store and validate
Write clean CSVs and SQLite tables and check the reference acceptance values.

In [9]:
tables = store(customers, products_raw, orders)
validate(tables['customers'], tables['products'], tables['orders'])


Saved clean CSVs and D:\Sowbarnika\Instamat_project\data\instamart.db
ALL CHECKPOINTS PASSED


## Task 6 — Seven SQL business queries
Execute each statement from `sql/queries.sql` against the persisted SQLite database.

In [10]:
sql_results = run_sql()



Q1 rows: 30
customer_id  qualifying_spend
      C0693          13776.55
      C0082          12317.70
      C0743          12266.85
      C0525          12241.30
      C0291          12150.70
      C0267          12150.25
      C0488          11825.50
      C0699          11693.90
      C0512          11617.65
      C0332          10779.50
      C0543          10636.10
      C0775          10263.55
      C1024          10160.10
      C0426           9884.60
      C0308           9872.40
      C1147           9740.80
      C0407           9451.75
      C0930           9194.00
      C0130           9170.40
      C0147           9141.50
      C1146           8993.05
      C1136           8983.75
      C0696           8941.75
      C0167           8718.55
      C0819           8695.30
      C0942           8643.60
      C1000           8462.20
      C0916           8395.35
      C0454           8299.15
      C0424           8294.20

Q2 rows: 30
     city      category   revenue
Bengaluru 

## Task 7 — pandas cross-check and EDA
Rebuild city/category revenue in pandas, confirm the SQL grand total, and examine completed-order distribution, outliers and correlations.

In [11]:
pivot, city_summary, distribution, correlation = pandas_eda(customers, products_raw, orders, sql_results)


Pandas pivot:
 category       Dairy  Electronics    Grocery  Personal Care     Snacks         All
city                                                                              
Bengaluru   87075.05    233909.45  110915.65       20193.85   71549.40   523643.40
Chennai     78301.80    195508.80   93817.45       27675.40   68325.70   463629.15
Delhi      117271.70    158549.10  112658.05       23471.50   90879.45   502829.80
Hyderabad   83731.00    228913.10  109475.40       24800.05   81670.35   528589.90
Mumbai      75209.85    214738.40   78970.15       38018.00   72122.10   479058.50
Pune        90349.55    187629.25  108456.60       34275.15   80207.50   500918.05
All        531938.95   1219248.10  614293.30      168433.95  464754.50  2998668.80

City summary:
                  sum        mean  count
city                                   
Bengaluru  523643.40  276.474868   1894
Chennai    463629.15  272.723029   1700
Delhi      502829.80  243.619089   2064
Hyderabad  528589.90  